In [3]:
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from utils import add_fractional_year
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score
from scipy.stats import randint, uniform
from catboost import CatBoostRegressor, Pool
from catboost.utils import get_roc_curve

# Data importing

In [60]:
X_train_1 =pd.read_csv("../data/boost/X_train_boost_part1.csv", index_col = "index")
X_train_2 =pd.read_csv("../data/boost/X_train_boost_part2.csv", index_col = "index")
X_train_3 =pd.read_csv("../data/boost/X_train_boost_part3.csv", index_col = "index")
X_train = pd.concat([X_train_1,X_train_2,X_train_3])

X_test = pd.read_csv("../data/boost/X_test_boost.csv", index_col = "index")

y_train = pd.read_csv("../data/y_train.csv", index_col = "index")
y_test = pd.read_csv("../data/y_test.csv", index_col = "index")


In [61]:
# changing categorical variables into correct type
categorical_cols = ['flat_type', 'town', 'flat_model', 'block', 'street_name' ] 

for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [ ]:
# handle date as it is not accepted by lightgbm

# try using year only
X_train_light = add_fractional_year(X_train, date_col='month', new_col='month_fraction')
X_train_light.drop(columns='month', inplace=True) 

X_test_light = add_fractional_year(X_test, date_col='month', new_col='month_fraction')
X_test_light.drop(columns='month', inplace=True) 

,town,flat_type,block,street_name,floor_area_sqm,flat_model,remaining_lease,month_fraction
index,,,,,,,,
545942,TAMPINES,EXECUTIVE,497J,TAMPINES ST 45,139.0,PREMIUM APARTMENT,86.7,2008.25000
444740,KALLANG/WHAMPOA,3 ROOM,463,CRAWFORD LANE,60.0,IMPROVED,76.2,2004.66667
638905,JURONG WEST,5 ROOM,987D,JURONG WEST ST 93,110.0,PREMIUM APARTMENT,93.6,2011.33333
295750,WOODLANDS,5 ROOM,877,WOODLANDS AVE 9,126.0,IMPROVED,94.7,2000.25000
829849,BUKIT MERAH,2 ROOM,28,JLN KLINIK,49.0,STANDARD,47.7,2020.91667
...,...,...,...,...,...,...,...,...
354890,QUEENSTOWN,3 ROOM,5,DOVER CRES,82.0,NEW GENERATION,76.1,2001.83333
903984,PASIR RIS,4 ROOM,190,PASIR RIS ST 12,106.0,MODEL A,69.2,2023.58333
485178,YISHUN,4 ROOM,390,YISHUN AVE 6,104.0,MODEL A,80.9,2006.00000


# LightGBM

In [5]:
# setting parameters
params = {
    'objective': 'regression',           # Type of task: regression
    'metric': 'rmse',                    # Root Mean Squared Error
    'boosting_type': 'gbdt',             # Gradient Boosting Decision Trees
    'learning_rate': 0.1,                # Step size shrinkage
    'num_leaves': 31,                    # Max leaf nodes per tree
    'max_depth': -1,                     # No limit (-1)
    'feature_fraction': 0.9,             # Randomly select 90% of features for each tree
    'bagging_fraction': 0.8,             # Randomly select 80% of data for each iteration
    'bagging_freq': 5,                   # Perform bagging every 5 iterations
    'verbose': 1                      
}

In [6]:
X_train_sub, X_valid, y_train_sub, y_valid = train_test_split(
    X_train_light, y_train, test_size=0.2, random_state=42
)

lgb_train = lgb.Dataset(X_train_sub, label=y_train_sub.values.ravel(), categorical_feature=categorical_cols if categorical_cols else 'auto')
lgb_valid = lgb.Dataset(X_valid, label=y_valid.values.ravel(), reference=lgb_train)

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=['train', 'valid'],
    num_boost_round=1000
)



[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019586 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3539
[LightGBM] [Info] Number of data points in the train set: 597519, number of used features: 8
[LightGBM] [Info] Start training from score 323716.792710


In [7]:
prediction = model.predict(X_test_light)
r2_score_test = r2_score(y_test, prediction)
print(f"R2 score for LightGBM test = {r2_score_test:.4f}")


R2 score for LightGBM test = 0.9820


## Hyperparameter tuning

In [11]:
model_light = LGBMRegressor(
    objective = 'regression',
    device = 'cpu'
)


# Define parameter space
param_dist = {
    'learning_rate': uniform(0.01, 0.3),
    'num_leaves': randint(20, 3000),
    'max_depth': randint(3, 15),
    'min_child_samples': randint(5, 100),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'reg_alpha': uniform(1e-8, 10),
    'reg_lambda': uniform(1e-8, 10)
}

search = RandomizedSearchCV(
    model_light,
    param_distributions=param_dist,
    n_iter = 50,
    cv = 5,
    scoring='r2',
    verbose = 2,
    n_jobs = -1
)

In [13]:
search.fit(X_train_light, y_train.values.ravel(), 
           eval_set=[(X_valid, y_valid.values.ravel())],
           eval_metric='rmse',
           categorical_feature=categorical_cols)

# Best model
best_model = search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022409 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3533
[LightGBM] [Info] Number of data points in the train set: 746899, number of used features: 8
[LightGBM] [Info] Start training from score 323626.670953
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [14]:
# Predict and evaluate
y_pred = best_model.predict(X_test_light)
r2 = r2_score(y_test, y_pred)

print("Best Params:", search.best_params_)
print(f"Test R²: {r2:.4f}")

Best Params: {'colsample_bytree': 0.8470992009988975, 'learning_rate': 0.20671647147554234, 'max_depth': 14, 'min_child_samples': 18, 'num_leaves': 1513, 'reg_alpha': 1.463579017403274, 'reg_lambda': 5.259107353374589, 'subsample': 0.9939773336291013}
Test R²: 0.9831


# Catboost

## Data

In [62]:
# changing year into numerical value
X_train = add_fractional_year(X_train, date_col='month', new_col='year')
X_test = add_fractional_year(X_test, date_col = 'month', new_col = 'year')

X_train.drop(columns = 'month', inplace = True)
X_test.drop(columns = 'month', inplace = True)

X_train_cat, X_valid_cat, y_train_cat, y_valid_cat = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [63]:
print(X_train.dtypes)

town               category
flat_type          category
block              category
street_name        category
floor_area_sqm      float64
flat_model         category
remaining_lease     float64
year                float64
dtype: object


In [64]:
# creating pools for catboost

train_pool = Pool(
    data = X_train_cat,
    label = y_train_cat.values.ravel(),
    cat_features=categorical_cols,
)

valid_pool = Pool(
    data=X_valid_cat,
    label=y_valid_cat.values.ravel(),
    cat_features=categorical_cols
)

test_pool = Pool(
    data=X_test,
    cat_features=categorical_cols
)

## Training

In [68]:
# using defult parameters

model = CatBoostRegressor(
    task_type="CPU", # for windows
    iterations=1000,
    learning_rate=0.1,
    depth = 5,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose = 100,
    early_stopping_rounds=50
)

In [69]:
model.fit(train_pool,eval_set=valid_pool, use_best_model=True)

# Evaluate
y_pred = model.predict(test_pool)
r2 = r2_score(y_test, y_pred)
print(f"CatBoost R² score on test set: {r2:.4f}")

0:	learn: 159737.4418120	test: 160292.9472835	best: 160292.9472835 (0)	total: 88.8ms	remaining: 1m 28s
100:	learn: 39340.7831629	test: 38928.2794707	best: 38928.2794707 (100)	total: 4.89s	remaining: 43.5s
200:	learn: 34614.6327089	test: 34020.9293963	best: 34020.9293963 (200)	total: 9.65s	remaining: 38.4s
300:	learn: 32508.3428343	test: 31870.6297964	best: 31870.6297964 (300)	total: 13.8s	remaining: 32.1s
400:	learn: 31129.7580782	test: 30545.0057866	best: 30545.0057866 (400)	total: 18.2s	remaining: 27.2s
500:	learn: 30204.1617047	test: 29625.7365982	best: 29625.7365982 (500)	total: 22.5s	remaining: 22.4s
600:	learn: 29475.4851107	test: 28911.8546106	best: 28911.8546106 (600)	total: 26.6s	remaining: 17.6s
700:	learn: 28925.5011790	test: 28398.6863617	best: 28398.6863617 (700)	total: 31s	remaining: 13.2s
800:	learn: 28442.7088490	test: 27940.4232625	best: 27940.4232625 (800)	total: 35.2s	remaining: 8.73s
900:	learn: 28094.5713482	test: 27613.5050172	best: 27613.5050172 (900)	total: 39.6

# Dimentionality Reduction
Trying to remove some of the features such as `Block` and `Steet Name` which may not be so necessary.

In [73]:
X_train_2 = X_train.drop(columns = ['block', 'street_name'])
X_test_2 = X_test.drop(columns = ['block', 'street_name'])


X_train_cb_2, X_valid_cb_2, y_train_cb_2, y_valid_cb_2 = train_test_split(
    X_train_2, y_train, test_size=0.2, random_state=42
)

categorical_cols_2=['flat_type','town','flat_model']

# creating pools for catboost

train_pool_2 = Pool(
    data = X_train_cb_2,
    label = y_train_cb_2.values.ravel(),
    cat_features=categorical_cols_2
)

valid_pool_2 = Pool(
    data=X_valid_cb_2,
    label=y_valid_cb_2.values.ravel(),
    cat_features=categorical_cols_2
)

test_pool_2 = Pool(
    data=X_test_2,
    cat_features=categorical_cols_2
)

In [74]:
model_small = CatBoostRegressor(
    task_type="CPU", # for mac
    iterations=1000,
    learning_rate=0.1,
    depth = 5,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose = 100,
    early_stopping_rounds=50
)

model_small.fit(train_pool_2,eval_set=valid_pool_2, use_best_model=True)

0:	learn: 159959.1708537	test: 160519.1276222	best: 160519.1276222 (0)	total: 74.8ms	remaining: 1m 14s
100:	learn: 41996.0883763	test: 41994.4937022	best: 41994.4937022 (100)	total: 3.43s	remaining: 30.6s
200:	learn: 36537.1894970	test: 36451.3593221	best: 36451.3593221 (200)	total: 6.93s	remaining: 27.5s
300:	learn: 34686.8941853	test: 34607.7169843	best: 34607.7169843 (300)	total: 10.1s	remaining: 23.6s
400:	learn: 33633.0002561	test: 33565.9615007	best: 33565.9615007 (400)	total: 13.2s	remaining: 19.7s
500:	learn: 32822.4429059	test: 32771.3648233	best: 32771.3648233 (500)	total: 16.2s	remaining: 16.1s
600:	learn: 32327.9090688	test: 32298.3931862	best: 32298.3931862 (600)	total: 19.1s	remaining: 12.7s
700:	learn: 31868.4568478	test: 31867.1697236	best: 31867.1697236 (700)	total: 22.1s	remaining: 9.42s
800:	learn: 31541.8131067	test: 31554.4702977	best: 31554.4702977 (800)	total: 25.3s	remaining: 6.27s
900:	learn: 31263.5860242	test: 31299.4153578	best: 31299.4153578 (900)	total: 28

In [75]:
# Evaluate
y_pred = model_small.predict(test_pool_2)
r2 = r2_score(y_test, y_pred)
print(f"CatBoost R² score on test set: {r2:.4f}")

CatBoost R² score on test set: 0.9684


The decrease is small. We will keep using this set of variables.

## Hyperparameter Tuning for smaller model


In [ ]:
cb_search = CatBoostRegressor(task_type="CPU", loss_function='RMSE', verbose=0, random_state=42)

param_dist = {
    'depth': randint(5, 15),
    'learning_rate': uniform(0.05, 0.2),
    'l2_leaf_reg': uniform(3, 10),
    'iterations': randint(500, 1500)
}

search_result = cb_search.randomized_search(
    param_dist,
    Pool(X_train_2, y_train.values.ravel(), cat_features=categorical_cols_2),
    n_iter=20,  # how many parameter sets to try
    cv=3,
    partition_random_seed=42,
    verbose=True,
    calc_cv_statistics=True,
    train_size=0.8
)

In [81]:
search_result['params']

{'depth': 11,
 'learning_rate': 0.24908503308204155,
 'l2_leaf_reg': 3.4458402159097217,
 'iterations': 1058}

In [83]:
final_model = CatBoostRegressor(
    task_type="CPU", # for mac
    iterations=1100,
    learning_rate=0.25,
    depth = 11,
    l2_leaf_reg= 3.45,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose = 100,
    early_stopping_rounds=50
)

final_model.fit(train_pool_2,eval_set=valid_pool_2, use_best_model=True)

0:	learn: 138418.9007021	test: 138910.5541388	best: 138910.5541388 (0)	total: 146ms	remaining: 2m 40s
100:	learn: 30036.6914491	test: 30264.9648281	best: 30264.9648281 (100)	total: 7.9s	remaining: 1m 18s
200:	learn: 28102.9887712	test: 28707.7099174	best: 28707.7099174 (200)	total: 15s	remaining: 1m 7s
300:	learn: 27116.1645123	test: 27995.6217828	best: 27995.6217828 (300)	total: 22.5s	remaining: 59.9s
400:	learn: 26457.5887109	test: 27619.5636082	best: 27619.5636082 (400)	total: 29.9s	remaining: 52.1s
500:	learn: 25938.1716356	test: 27364.9250750	best: 27364.9250750 (500)	total: 37.2s	remaining: 44.5s
600:	learn: 25498.7276291	test: 27170.4836862	best: 27170.4836862 (600)	total: 44.8s	remaining: 37.2s
700:	learn: 25169.2432938	test: 27051.5171853	best: 27051.5171853 (700)	total: 53s	remaining: 30.2s
800:	learn: 24869.0455290	test: 26943.8758720	best: 26943.8758720 (800)	total: 1m 1s	remaining: 22.9s
900:	learn: 24614.1006607	test: 26859.7590572	best: 26859.7590572 (900)	total: 1m 9s	r

In [84]:
# Evaluate
y_pred = final_model.predict(test_pool_2)
r2 = r2_score(y_test, y_pred)
print(f"CatBoost R² score on test set: {r2:.4f}")

CatBoost R² score on test set: 0.9764


In [85]:
final_model.save_model("../model/catboost.cbm")

In [110]:
# Create test row
data = pd.DataFrame([{
    'town': 'ANG MO KIO',
    'flat_type': '4 ROOM',
    'floor_area_sqm': 90,
    'flat_model': 'MODEL A',
    'remaining_lease': 50,
    'year': 2090,
}])

# Make sure categorical columns are of category dtype
for col in ['town', 'flat_model', 'flat_type']:
    data[col] = data[col].astype('category')

# Create Pool for prediction
test_pool = Pool(
    data=data,
    cat_features=['town', 'flat_model', 'flat_type']
)

In [111]:
final_model.predict(test_pool)

array([520602.54012176])